# Directionality: Fuzzy vs Crisp (RecurrenceTime)

We compute association rules and keep the top-10 by confidence.
We do NOT prune to 1->1 rules, but we compute asymmetry only for 1->1 rules inside the top-10 (if any).

Settings:
- Fuzzy (Lukasiewicz t-norm, custom F)
- Fuzzy (Lukasiewicz t-norm, F = T_luk = ignore implication)
- Crisp (discretized by max membership)


In [242]:
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

import FIRM.base.fuzzy_data as fuzzy_data
from FIRM.methods.AARFI import AARFI_F


In [243]:
# -------------------------
# Config
# -------------------------
DATASET_PATH = "../assets/wdbc_rt.csv"
FILTER_FEATURE = "RecurrenceTime"  # keep only rules that include this feature
TOP_K = 10
CORR_CUTOFF = 0.95  # drop features with |corr| >= this
KEEP_FEATURE = FILTER_FEATURE  # never drop this feature

N_LABELS = 3
MIN_COV = 0.2
MIN_SUPP = 0.2
MIN_CONF = 0.65
MAX_FEAT = 2

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 0)


In [244]:
def process_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        s = out[col]
        if pd.api.types.is_integer_dtype(s):
            out[col] = s.astype("float64")
            continue
        is_cat_like = (
            pd.api.types.is_object_dtype(s)
            or isinstance(s.dtype, pd.CategoricalDtype)
            or pd.api.types.is_string_dtype(s)
        )
        if is_cat_like:
            out[col] = s.astype("object")
    return out


def drop_high_corr_features(df: pd.DataFrame, cutoff: float, keep_feature: str = None) -> pd.DataFrame:
    num = df.select_dtypes(include=[np.number])
    if num.shape[1] <= 1:
        return df
    corr = num.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] >= cutoff)]
    if keep_feature and keep_feature in to_drop:
        to_drop = [c for c in to_drop if c != keep_feature]
    return df.drop(columns=to_drop)
def token_to_feature(token: str) -> str:
    return token.rsplit("_", 1)[0]

def tuple_to_token(item, dataset, fuzzy_dataset):
    fi, li = item
    return f"{dataset.columns[fi]}_{fuzzy_dataset.fv_list[fi].get_labels[li]}"

def rule_str(ants, con):
    if len(ants) == 0:
        return f"-> {con}"
    return " & ".join(ants) + " -> " + con

def filter_by_feature(df: pd.DataFrame, feature: str) -> pd.DataFrame:
    if not feature:
        return df
    ant_has = df["ant_tokens"].apply(
        lambda xs: feature in {token_to_feature(t) for t in xs}
    )
    con_has = df["con_token"].apply(
        lambda t: token_to_feature(t) == feature
    )
    return df[ant_has | con_has]
def add_asymmetry_top10(df_all: pd.DataFrame, df_top: pd.DataFrame) -> pd.DataFrame:
    out = df_top.copy()
    # Build lookup for 1->1 rules
    one_one = df_all[df_all["n_antecedents"] == 1]
    lookup = {(r.ant_tokens[0], r.con_token): (r.confidence, r.support) for r in one_one.itertuples(index=False)}
    swap_conf = []
    swap_supp = []
    abs_diff_support = []
    for r in out.itertuples(index=False):
        if r.n_antecedents == 1:
            key = (r.con_token, r.ant_tokens[0])
            val = lookup.get(key)
            if val is None:
                swap_conf.append(np.nan)
                swap_supp.append(np.nan)
                abs_diff_support.append(np.nan)
            else:
                sc, ss = val
                swap_conf.append(sc)
                swap_supp.append(ss)
                abs_diff_support.append(abs(r.support - ss))
        else:
            swap_conf.append(np.nan)
            swap_supp.append(np.nan)
            abs_diff_support.append(np.nan)
    out["swap_confidence"] = swap_conf
    out["swap_support"] = swap_supp
    out["abs_diff_support"] = abs_diff_support
    return out

def top10_by_confidence(df: pd.DataFrame, top_k: int) -> pd.DataFrame:
    if df.empty:
        return df
    return df.sort_values(["confidence", "support", "coverage"], ascending=False).head(top_k).reset_index(drop=True)


In [245]:
# -------------------------
# Load and fuzzify
# -------------------------
df_raw = pd.read_csv(DATASET_PATH)
dataset = process_df(df_raw)
dataset = drop_high_corr_features(dataset, CORR_CUTOFF, keep_feature=KEEP_FEATURE)

labels = ["L", "M", "H"] if N_LABELS == 3 else [f"L{i+1}" for i in range(N_LABELS)]
fuzzy_dataset = fuzzy_data.FuzzyDataQuantiles("symmetric", dataset, N_LABELS, labels)
print(f"rows={len(dataset)}, cols={len(dataset.columns)}")


rows=47, cols=11


In [246]:
# -------------------------
# Fuzzy settings
# -------------------------
def t_luk(x, y):
    return np.maximum(x + y - 1, 0)

def t_prod(x, y):
    return x * y

F_custom = lambda x, y: x * (y ** 0.01)

def run_fuzzy(name, T, F):
    _, measures = AARFI_F(
        dataset,
        fuzzy_dataset,
        T=T,
        I=None,
        F=F,
        min_cov=MIN_COV,
        min_supp=MIN_SUPP,
        min_conf=MIN_CONF,
        max_feat=MAX_FEAT,
        verbose=False,
    )

    rows = []
    for _, r in measures.iterrows():
        lrule = list(r["lrule"])
        if len(lrule) < 2:
            continue
        ant_items = lrule[:-1]
        con_item = lrule[-1]
        ant_tokens = [tuple_to_token(a, dataset, fuzzy_dataset) for a in ant_items]
        con_token = tuple_to_token(con_item, dataset, fuzzy_dataset)
        rows.append({
            "rule": rule_str(ant_tokens, con_token),
            "ant_tokens": ant_tokens,
            "con_token": con_token,
            "n_antecedents": len(ant_tokens),
            "coverage": float(r["coverage"]),
            "support": float(r["support"]),
            "confidence": float(r["confidence"]),
        })

    df = pd.DataFrame(rows)
    df = filter_by_feature(df, FILTER_FEATURE)
    top10 = top10_by_confidence(df, TOP_K)
    top10 = add_asymmetry_top10(df, top10)

    print(f"{name}: total rules={len(df)}")
    display(top10[["rule", "confidence", "support", "coverage", "n_antecedents", "swap_confidence", "swap_support", "abs_diff_support"]])
    return df, top10

fuzzy_custom_df, fuzzy_custom_top = run_fuzzy("fuzzy_luk_T_custom_F", t_luk, F_custom)
fuzzy_ignore_df, fuzzy_ignore_top = run_fuzzy("fuzzy_prod_T_prod_F", t_prod, t_prod)


fuzzy_luk_T_custom_F: total rules=7


,rule,confidence,support,coverage,n_antecedents,swap_confidence,swap_support,abs_diff_support
0,Radius_L -> RecurrenceTime_H,0.705747,0.257698,0.365142,1,0.656760,0.243346,0.014352
1,SizeTumor_H -> RecurrenceTime_H,0.688105,0.233962,0.340009,1,NaN,NaN,NaN
2,RecurrenceTime_L -> Radius_H,0.678060,0.284414,0.419453,1,0.675979,0.242050,0.042364
3,Radius_H -> RecurrenceTime_L,0.675979,0.242050,0.358074,1,0.678060,0.284414,0.042364
4,Fractality_H -> RecurrenceTime_H,0.671985,0.233460,0.347418,1,NaN,NaN,NaN
5,RecurrenceTime_H -> Radius_L,0.656760,0.243346,0.370524,1,0.705747,0.257698,0.014352
6,RecurrenceTime_H -> ConcavePoints_L,0.655221,0.242775,0.370524,1,NaN,NaN,NaN


fuzzy_prod_T_prod_F: total rules=0


,rule,confidence,support,coverage,n_antecedents,swap_confidence,swap_support,abs_diff_support


In [247]:
# -------------------------
# Crisp setting
# -------------------------
data_crisp = dataset.copy()
for i in range(len(fuzzy_dataset.fv_list)):
    data_crisp[dataset.columns[i]] = dataset[dataset.columns[i]].map(
        lambda x: fuzzy_dataset.fv_list[i].eval_max_fuzzy_set(x)
    )

df_encoded = pd.get_dummies(data_crisp, columns=data_crisp.columns)
df_freq = apriori(
    df_encoded,
    min_support=MIN_COV,
    use_colnames=True,
    verbose=0,
    max_len=MAX_FEAT + 1,
    low_memory=True,
)
df_ar = association_rules(df_freq, metric="confidence", min_threshold=MIN_CONF)

rows = []
for _, r in df_ar.iterrows():
    ants = sorted(list(r["antecedents"]))
    cons = sorted(list(r["consequents"]))
    if len(cons) != 1:
        continue
    ant_tokens = [str(a) for a in ants]
    con_token = str(cons[0])
    rows.append({
        "rule": rule_str(ant_tokens, con_token),
        "ant_tokens": ant_tokens,
        "con_token": con_token,
        "n_antecedents": len(ant_tokens),
        "coverage": float(r["antecedent support"]),
        "support": float(r["support"]),
        "confidence": float(r["confidence"]),
    })

crisp_df = pd.DataFrame(rows)
crisp_df = filter_by_feature(crisp_df, FILTER_FEATURE)
crisp_top = top10_by_confidence(crisp_df, TOP_K)
crisp_top = add_asymmetry_top10(crisp_df, crisp_top)

print(f"crisp: total rules={len(crisp_df)}")
display(crisp_top[["rule", "confidence", "support", "coverage", "n_antecedents", "swap_confidence", "swap_support", "abs_diff_support"]])


crisp: total rules=1


,rule,confidence,support,coverage,n_antecedents,swap_confidence,swap_support,abs_diff_support
0,Radius_H -> RecurrenceTime_L,0.666667,0.212766,0.319149,1,NaN,NaN,NaN
